In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_gtfs_calendar = spark.table("bg_traffic.bg_traffic_bronze.gtfs_calendar")
bronze_gtfs_calendar.display()

In [0]:
required_columns = {
    "service_id", "monday", "tuesday", "wednesday", "thursday", 
    "friday", "saturday", "sunday", "start_date", "end_date"
}
missing_columns = required_columns - set(bronze_gtfs_calendar.columns)
if missing_columns:
    raise ValueError("GRESKA: Izvorni GTFS calendar je promenio strukturu, postoje nedostajuce kolone!")

In [0]:
bronze_gtfs_calendar.printSchema()

In [0]:

bronze_gtfs_calendar.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in bronze_gtfs_calendar.columns]).show()

### Casting

In [0]:
types_calednar = bronze_gtfs_calendar.select(
    F.col("service_id").cast("string"),
    F.col("monday").cast("integer"),
    F.col("tuesday").cast("integer"),
    F.col("wednesday").cast("integer"),
    F.col("thursday").cast("integer"),
    F.col("friday").cast("integer"),
    F.col("saturday").cast("integer"),
    F.col("sunday").cast("integer"),
    F.col("start_date").cast("integer"),
    F.col("end_date").cast("integer")
)

types_calednar.display()

### Dedup

In [0]:
dedup_calendar = types_calednar.dropDuplicates(['service_id'])
dedup_count = types_calednar.count() - dedup_calendar.count()
print(f"Broj duplikata: {dedup_count}")

### Valid

In [0]:
valid_calendar = dedup_calendar.filter(
    (F.col("service_id").isNotNull()) 
).withColumn(
    "silver_processed_at", F.current_timestamp()
)
valid_calendar.display()

In [0]:
if valid_calendar.isEmpty():
    raise Exception("GRESKA: Silver tabela za upisivanje je prazna nakon ciscenja!")

### Write in silver table

In [0]:
valid_calendar.write.format("delta").mode("overwrite").option(
    "overwriteSchema","true"
).saveAsTable("bg_traffic.bg_traffic_silver.gtfs_calendar")